# ДЗ9 · Ranker · Этап 3
Colab T4. В Google Drive в папке HW9_Ranker разместите bundle.zip и четыре CSV в FAR-Trans/. Запускайте ячейки сверху вниз. Журналы хранятся в Drive. Для продолжения используйте ту же папку, тот же бандл и то же окружение. Два экземпляра блокнота одновременно не запускайте.

In [1]:
# 1. Drive and immutable paths.
from pathlib import Path
import os
import sys

DRY_RUN = False
if DRY_RUN:
    workspace = Path(os.environ["HW9_DRY_WORKSPACE"])
    bundle = workspace / "bundle"
    results = workspace / "dry_results"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    workspace = Path("/content/drive/MyDrive/HW9_Ranker")
    bundle = Path("/content/hw9_bundle")
    results = workspace / "colab_results"
archive = workspace / "bundle.zip"
data_dir = workspace / "FAR-Trans"
results.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [2]:
# 2. Verify archive and all its members before importing bundled code.
import hashlib
import json
from zipfile import ZipFile

def file_hash(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

expected_archive = "f3806321f4c3d36f69783be8d0a61934b7ec68f1062993de26be8aec1e5e3fc1"
assert file_hash(archive) == expected_archive, "Wrong bundle.zip"
with ZipFile(archive) as zipped:
    manifest = json.loads(zipped.read("manifest.json"))
    files = manifest["file_sha256"]
    assert sorted(zipped.namelist()) == sorted([*files, "manifest.json"])
    assert all(not Path(name).is_absolute() and ".." not in Path(name).parts for name in files)
    for filename, expected in files.items():
        assert hashlib.sha256(zipped.read(filename)).hexdigest() == expected, filename
    zipped.extractall(bundle)

sys.path.insert(0, str(bundle))
from stage3_runtime import SETS, append, check_identity, dry_run, read_jsonl, run_set
gen = manifest["generation"]
if DRY_RUN:
    print(dry_run(bundle, results))
else:
    shared = json.loads((bundle / "protocol.json").read_bytes())
    assert shared["protocol_hash"] == manifest["protocol_hash"]
    for filename, expected in shared["source_sha256"].items():
        assert file_hash(data_dir / filename) == expected, filename
    print("Bundle and four external CSV verified.")

Bundle and four external CSV verified.


In [3]:
# 3. Hardware and HW8 dependency pins. Keep Colab-provided torch.
if not DRY_RUN:
    import subprocess
    hardware = subprocess.run(["nvidia-smi"], check=True, capture_output=True, text=True).stdout
    print(hardware)
    (results / "nvidia-smi.log").write_text(hardware, encoding="utf-8")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6",
                    "huggingface_hub==0.36.0", "hf_xet==1.2.0", "psutil==7.2.2"], check=True)
    import torch
    import transformers
    assert torch.cuda.is_available()
    gpu = torch.cuda.get_device_name()
    assert "T4" in gpu, f"Select Colab T4, current GPU: {gpu}"
    native_bf16 = torch.cuda.get_device_capability()[0] >= 8
    print({"gpu": gpu, "torch": torch.__version__, "transformers": transformers.__version__,
           "native_bfloat16": native_bf16})

Thu Nov 26 20:09:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# 4. Pin the checkpoint and verify all hashes, including weight shards.
if not DRY_RUN:
    from huggingface_hub import snapshot_download
    import time
    started = time.perf_counter()
    weights = Path(snapshot_download(repo_id=gen["model"], revision=gen["revision"],
                                     allow_patterns=list(manifest["model_files_sha256"]), token=False))
    assert weights.name == gen["revision"]
    weight_hashes = {name: file_hash(weights / name) for name in manifest["model_files_sha256"]}
    assert weight_hashes == manifest["model_files_sha256"]
    identity = {"bundle_sha256": expected_archive, "revision": gen["revision"], "gpu": gpu,
                "torch": torch.__version__, "transformers": transformers.__version__,
                "native_bfloat16": native_bf16, "generation": gen, "weight_sha256": weight_hashes}
    check_identity(results / "run_identity.json", identity)
    append(results / "timing.jsonl", {"stage": "checkpoint_and_verification",
                                     "seconds": time.perf_counter() - started})
    print("Checkpoint verified:", weights.name)

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (2 outputs left out)

model-00001-of-00002.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Checkpoint verified: 70d244cc86ccca08cf5af4e1e306ecf908b1ad5e


In [5]:
# 5. Main, then ablation 0, 1 and 2. Raw response strings are not parsed.
if not DRY_RUN:
    from gpu_llm import GPUQwen
    import traceback

    started = time.perf_counter()
    backend = GPUQwen(weights, gen)
    append(results / "timing.jsonl", {"stage": "main_model_load", "seconds": time.perf_counter() - started,
                                     "gpu": gpu, "dtype": backend.metadata["dtype"]})
    try:
        for name in SETS:
            prompts = read_jsonl(bundle / f"{name}.jsonl")
            assert len(prompts) == manifest["prompt_counts"][name] == 100
            journal = results / f"{name}.jsonl"
            previous = len(read_jsonl(journal))
            append(results / "timing.jsonl", {"stage": name, "event": "start", "completed": previous,
                                             "resumed": previous > 0})
            try:
                summary = run_set(prompts, journal, backend)
            finally:
                recorded = read_jsonl(journal)
                append(results / "timing.jsonl", {"stage": name, "event": "end", "completed": len(recorded),
                                                 "new": len(recorded) - previous,
                                                 "new_generation_seconds": sum(x["seconds"] for x in recorded[previous:])})
            print(name, summary)
    except Exception:
        with (results / "main_errors.log").open("a", encoding="utf-8") as stream:
            traceback.print_exc(file=stream)
        raise
    finally:
        del backend
        import gc
        gc.collect()
        torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

main {'skipped': 0, 'new': 100, 'generation_seconds': 821.3869263739996}
ablation_seed0 {'skipped': 0, 'new': 100, 'generation_seconds': 836.9315113310008}
ablation_seed1 {'skipped': 0, 'new': 100, 'generation_seconds': 830.1983517179997}
ablation_seed2 {'skipped': 0, 'new': 100, 'generation_seconds': 845.8809817409992}


In [6]:
# 6. Teacher ranker: unchanged prompt, runtime, parser and fallback, with labels and scoring deferred.
if not DRY_RUN:
    assert all(len(read_jsonl(results / f"{name}.jsonl")) == 100 for name in SETS)
    teacher = results / "teacher"
    teacher.mkdir(exist_ok=True)
    if (teacher / "run_manifest.json").exists():
        assert len(read_jsonl(teacher / "raw_generations.jsonl")) == 100
        print("Teacher comparator already complete.")
    else:
        command = [sys.executable, "-B", str(bundle / "stage3_teacher.py"),
                   "--data-dir", str(data_dir), "--output-dir", str(teacher),
                   "--cutoff", "2021-06-01", "--max-users", "100", "--candidate-count", "50",
                   "--on-invalid", "fallback", "--device", "cuda", "--model", gen["model"],
                   "--revision", gen["revision"]]
        with (teacher / "teacher.log").open("a", encoding="utf-8") as log:
            code = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT,
                                  env=dict(os.environ, PYTHONHASHSEED="0")).returncode
        if code:
            raise RuntimeError(f"Teacher stopped with exit {code}; see teacher/teacher.log. "
                               "Keep partial outputs. Do not change its dtype or retry completed IDs.")
        info = json.loads((teacher / "run_manifest.json").read_bytes())["runtime"]
        print({key: info[key] for key in ("actual_model_dtype", "device", "gpu",
                                         "model_load_seconds", "generation_seconds")})

{'actual_model_dtype': 'torch.bfloat16', 'device': 'cuda', 'gpu': 'Tesla T4', 'model_load_seconds': 27.96266727900047, 'generation_seconds': 2154.9845165119996}


In [7]:
# 7. Archive logs and responses. This cell also works after an interrupted run.
if not DRY_RUN:
    from google.colab import files as colab_files
    from zipfile import ZIP_DEFLATED
    download = workspace / "hw9_stage3_colab_results.zip"
    with ZipFile(download, "w", ZIP_DEFLATED) as zipped:
        for source in sorted(results.rglob("*")):
            if source.is_file() and source.suffix in {".json", ".jsonl", ".csv", ".log"}:
                zipped.write(source, source.relative_to(results))
    colab_files.download(str(download))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>